# 04 — Consultas al índice de Vector Search

Este notebook demuestra cómo consultar el índice `proyecto_mna.fichas_busqueda.fichas_vs_index` para recuperar fichas de personas desaparecidas por similitud semántica (k-NN), con y sin filtros de blocking. Forma parte de la fase de Modelado del CRISP-ML del Proyecto Integrador MNA.

In [0]:
# ============================================================================
# Cell 1 — Setup y configuración
# ============================================================================
import json
import pandas as pd
from databricks.sdk import WorkspaceClient

# --- Cliente del workspace (autenticación automática del notebook) ---
w = WorkspaceClient()

# --- Configuración del índice de Vector Search ---
# Índice: proyecto_mna.fichas_busqueda.texto_completo_sin_features
# Endpoint: forensic (ONLINE)
# Source: fichas_texto_embedding (texto_completo con descripciones detalladas de tatuajes)
# Primary key: id_ficha
# Embedding column: texto_completo (managed embeddings)
# Filter columns: sexo (HOMBRE/MUJER), edad_desaparicion (numeric), estado_hechos, municipio_hechos
INDEX_NAME = "proyecto_mna.fichas_busqueda.texto_completo_sin_features"
COLUMNS = [
    "id_ficha",
    "texto_completo",
    "texto_tatuajes",
    "texto_rasgos",
    "texto_senas",
    "texto_vestimenta",
    "sexo",
    "edad_desaparicion",
    "estado_hechos",
    "municipio_hechos",
]

print("✅ Configuración lista")
print(f"   Índice: {INDEX_NAME}")
print(f"   Columnas: {COLUMNS}")

In [0]:
# ============================================================================
# Cell 2 — Búsqueda k-NN básica (sin filtros)
# ============================================================================
# Edita query_text con la descripción de la persona que quieres buscar.
# El formato debe coincidir con el texto canónico de la tabla gold:
# "sexo ...; complexión ...; piel ...; cara ...; cabello ...; ojos ...; ..."
query_text = (
    "Hombre 50 a 60 años . Tatuaje monocromático a tinta negra, con la figura de la silueta de una cruz, localizado en región esternal. . Tatuaje monocromático a tinta negra, con la figura de un rostro de mujer, localizado en brazo derecho cara posterior tercio proximal. Tatuaje monocromático a tinta negra, con la figura de una silueta de la Virgen de Guadalupe, localizado en muslo derecho cara anterior tercio medio y distal."
)

print(f"Query: {query_text[:120]}...\n")

# query_index con query_type="ANN" hace búsqueda por similitud semántica.
# El modelo embebe query_text y compara contra todos los vectores del índice.
results = w.vector_search_indexes.query_index(
    index_name=INDEX_NAME,
    columns=COLUMNS,
    query_text=query_text,
    num_results=5,
    query_type="ANN",
    score_threshold=0,
)

# --- Mostrar resultados de forma legible ---
print("=" * 70)
print("RESULTADOS — k-NN sin filtros (top 5)")
print("=" * 70)
manifest = [c.name for c in results.manifest.columns]
for i, row in enumerate(results.result.data_array):
    score = row[-1]
    id_ficha = row[manifest.index("id_ficha")]
    sexo = row[manifest.index("sexo")]
    edad = row[manifest.index("edad_desaparicion")]
    estado = row[manifest.index("estado_hechos")]
    texto = row[manifest.index("texto_tatuajes")]
    print(f"  [{i+1}] score={score:.4f} | id={id_ficha} | {sexo} | edad={edad} | {estado}")
    print(f"      tatuajes: {texto[:100]}...")
    print()

In [0]:
# ============================================================================
# Cell 4 — Función reutilizable: buscar_candidatos
# ============================================================================
# Esta función está lista para integrarse en el notebook 04 del pipeline
# CRISP-ML para la fase de Modelado. Encapsula la consulta al índice y devuelve
# un DataFrame de pandas con los resultados ordenados por score.

def buscar_candidatos(query_text, k=5, filters=None):
    """
    Busca fichas candidatas por similitud semántica en el índice de Vector Search.

    Parámetros
    ----------
    query_text : str
        Descripción canónica de la persona a buscar (mismo formato que texto_canonico_fe).
        Ejemplo: "sexo hombre; complexión mediana; piel moreno; ..."
    k : int, opcional (default=5)
        Número de candidatos a retornar (top-k).
    filters : dict, opcional (default=None)
        Diccionario con filtros de blocking. Claves válidas:
        sexo (HOMBRE/MUJER), edad_desaparicion (numérico), estado_hechos, municipio_hechos.
        Si es None, no se aplican filtros.

    Retorna
    -------
    pd.DataFrame
        DataFrame con columnas del índice + score.
        Ordenado por score descendente.
    """
    resp = w.vector_search_indexes.query_index(
        index_name=INDEX_NAME,
        columns=COLUMNS,
        query_text=query_text,
        num_results=k,
        query_type="ANN",
        filters_json=json.dumps(filters) if filters else None,
        score_threshold=0,
    )
    cols = [c.name for c in resp.manifest.columns]
    rows = resp.result.data_array
    return pd.DataFrame(rows, columns=cols).sort_values("score", ascending=False)


# --- Ejemplo de uso ---
df_result = buscar_candidatos(
    query_text=(
        "sexo mujer; complexión delgada; piel moreno; cara ovalada; "
        "cabello negro mediano; ojos cafes oscuros medianos; nariz recta; "
        "boca mediana; labios delgados; estatura 160 cm; peso 50 kg; "
        "edad 25 años"
    ),
    k=5,
    filters={"sexo": "MUJER"},
)

print("=" * 70)
print("RESULTADOS — buscar_candidatos (MUJER, top 5)")
print("=" * 70)
display(df_result[["score", "id_ficha", "sexo", "edad_desaparicion", "estado_hechos"]])

In [0]:
# ============================================================================
# Cell 5 — Comparación: DESCRIPCIÓN TATUAJES (necropsias) vs índice Vector Search
# ============================================================================
# Carga la columna DESCRIPCIÓN TATUAJES de la tabla de necropsias y, 
# para cada registro con tatuajes reales, llama a buscar_candidatos() 
# para encontrar fichas de personas desaparecidas por similitud semántica.

# --- Cargar descripciones de tatuajes de la tabla de necropsias ---
necropsias_df = spark.sql("""
    SELECT 
        `#` AS id_necropsia,
        SEXO AS sexo_necropsia,
        `EDAD (AÑOS, MESES, SEMANAS, DÍAS)` AS edad_necropsia,
        `DESCRIPCIÓN TATUAJES` AS descripcion_tatuajes
    FROM proyecto_mna.fichas_busqueda.registro_necropsias_2026
    WHERE TATUAJES = 'SI'
      AND `DESCRIPCIÓN TATUAJES` IS NOT NULL
      AND `DESCRIPCIÓN TATUAJES` NOT IN ('NO APLICA', 'N/A', 'NO', '')
""")

# Collect al driver (dataset pequeño: 8 filas)
necropsias_rows = necropsias_df.collect()

print(f"Registros de necropsias con tatuajes: {len(necropsias_rows)}")
print("=" * 80)

# --- Para cada necropsia, buscar candidatos con blocking por sexo ---
todas_las_comparaciones = []

for row in necropsias_rows:
    query = row["descripcion_tatuajes"]
    id_nec = row["id_necropsia"]
    sexo_nec = row["sexo_necropsia"]
    edad_nec = row["edad_necropsia"]

    # Blocking directo por sexo (HOMBRE/MUJER — mismo formato del índice)
    filters = {"sexo": sexo_nec} if sexo_nec in ("HOMBRE", "MUJER") else None
    print(f"\n{'─' * 80}")
    print(f"Necropsia #{id_nec} | {sexo_nec} | {edad_nec}")
    print(f"Tatuajes: {query[:150]}...")
    print(f"Filtros de blocking: {filters if filters else 'Ninguno'}")

    # Búsqueda semántica con blocking por sexo y grupo de edad
    df_result = buscar_candidatos(query_text=query, k=5, filters=filters if filters else None)

    print(f"\n  Top 5 fichas candidatas:")
    for rank, (_, r) in enumerate(df_result.iterrows(), 1):
        score = r["score"]
        id_ficha = r["id_ficha"]
        sexo_ficha = r["sexo"]
        edad_ficha = r["edad_desaparicion"]
        estado_ficha = r["estado_hechos"]
        texto_tatuajes_ficha = str(r["texto_tatuajes"])

        print(f"    [{rank}] score={score:.4f} | id_ficha={id_ficha} | {sexo_ficha} | edad={edad_ficha} | {estado_ficha}")
        print(f"        tatuajes: {texto_tatuajes_ficha[:120]}...")

        todas_las_comparaciones.append({
            "id_necropsia": id_nec,
            "sexo_necropsia": sexo_nec,
            "edad_necropsia": edad_nec,
            "descripcion_tatuajes": query[:200],
            "rank": rank,
            "score": score,
            "id_ficha": id_ficha,
            "sexo_ficha": sexo_ficha,
            "edad_ficha": edad_ficha,
            "estado_ficha": estado_ficha,
            "texto_tatuajes_ficha": texto_tatuajes_ficha[:200],
        })

# --- Join con fichas_persona_silver para obtener la ruta de la imagen ---
df_comparacion = pd.DataFrame(todas_las_comparaciones)

fichas_info = spark.sql("""
    SELECT
        id_ficha,
        folio_unico,
        sexo,
        edad_desaparicion,
        estado_hechos,
        municipio_hechos,
        ruta
    FROM proyecto_mna.fichas_busqueda.fichas_persona_silver
""").toPandas()

df_comparacion = df_comparacion.merge(
    fichas_info,
    left_on="id_ficha",
    right_on="id_ficha",
    how="left",
    suffixes=("", "_silver"),
)

# --- Tabla comparativa consolidada ---
print("\n\n" + "=" * 80)
print("TABLA COMPARATIVA — Necropsias vs Fichas (Vector Search)")
print("=" * 80)
display(df_comparacion[["id_necropsia", "sexo_necropsia", "edad_necropsia",
                         "rank", "score", "id_ficha", "folio_unico",
                         "sexo_ficha", "edad_ficha", "estado_ficha",
                         "ruta"]])

In [0]:
# ============================================================================
# Cell 6 — Galería visual: fichas candidatas vs necropsias
# ============================================================================
# Muestra la imagen de cada ficha candidata almacenada en el volumen,
# agrupada por necropsia, para inspección visual del cotejo.

import base64
import os
import io
from PIL import Image as PILImage

# --- Función auxiliar: leer imagen del volumen como base64 (redimensionada) ---
def leer_imagen_b64(ruta, max_width=200):
    """Lee una imagen del volumen FUSE, la redimensiona y la codifica en base64."""
    if not ruta or not os.path.exists(ruta):
        return None
    ext = os.path.splitext(ruta)[1].lower()
    if ext not in (".jpeg", ".jpg", ".png"):
        return None  # Skip PDFs y otros formatos
    try:
        img = PILImage.open(ruta)
        img.thumbnail((max_width, max_width * 2))
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=70)
        return base64.b64encode(buf.getvalue()).decode()
    except Exception:
        return None

# --- Construir galería HTML por necropsia ---
for id_nec in sorted(df_comparacion["id_necropsia"].unique()):
    subset = df_comparacion[df_comparacion["id_necropsia"] == id_nec].sort_values("rank")
    nec_row = subset.iloc[0]

    html = f"<h3>Necropsia #{id_nec} — {nec_row['sexo_necropsia']} | {nec_row['edad_necropsia']}</h3>"
    html += f"<p style='color:#555;'><b>Tatuajes:</b> {nec_row['descripcion_tatuajes'][:250]}...</p>"
    html += "<div style='display:flex; flex-wrap:wrap; gap:12px; margin-bottom:20px;'>"

    for _, r in subset.iterrows():
        ruta = r["ruta"]
        score = r["score"]
        folio = str(r["folio_unico"]) if pd.notna(r["folio_unico"]) else "N/A"
        img_b64 = leer_imagen_b64(ruta)

        if img_b64:
            html += f"""
            <div style='text-align:center; border:1px solid #ccc; padding:8px; border-radius:8px; width:170px;'>
                <img src="data:image/jpeg;base64,{img_b64}" style="width:150px; height:auto; border-radius:4px;"/>
                <p style='font-size:11px; margin:4px 0;'><b>Rank {r['rank']}</b> | Score: {score:.4f}</p>
                <p style='font-size:9px; color:#777;'>{folio[:25]}</p>
                <p style='font-size:9px; color:#777;'>{r['sexo_ficha']} | edad={r['edad_ficha']} | {r['estado_ficha']}</p>
            </div>
            """
        else:
            ext = os.path.splitext(str(ruta))[1].lower() if ruta else ""
            label = "Documento PDF" if ext == ".pdf" else "Sin imagen"
            html += f"""
            <div style='text-align:center; border:1px solid #ccc; padding:8px; border-radius:8px; width:170px;'>
                <div style='width:150px; height:200px; background:#f0f0f0; display:flex; align-items:center; justify-content:center; border-radius:4px;'>
                    <span style='color:#999; font-size:12px;'>{label}</span>
                </div>
                <p style='font-size:11px; margin:4px 0;'><b>Rank {r['rank']}</b> | Score: {score:.4f}</p>
                <p style='font-size:9px; color:#777;'>{folio[:25]}</p>
            </div>
            """

    html += "</div><hr/>"
    displayHTML(html)